# 06 — Participant ratings

Reproduces Table S9: participant ratings of relevance, usefulness, and
disruptiveness by intervention arm, with Kruskal-Wallis omnibus tests
(Benjamini-Hochberg corrected across the three rating dimensions within each
experiment) and Dunn-Holm post-hoc tests for significant dimensions.

In [ ]:
import pandas as pd
from scipy.stats import kruskal, false_discovery_control
import scikit_posthocs as sp

ARM_COL = "CONTROL_GROUP"
ARM_MAP = {"AA": "A", "BB": "B", "CC": "C", "EE": "E", "FF": "F", "CONTROLGROUP": "Control"}
RATINGS = {"RATING_R": "Relevance", "RATING_U": "Usefulness", "RATING_D": "Disruptiveness"}

FILES = {
    "Experiment 1": {"path": "P10_final.csv", "arms": ["A", "C", "E", "F"]},
    "Experiment 2": {"path": "P11_final.csv", "arms": ["A", "B", "C"]},
}

In [ ]:
for exp_name, info in FILES.items():
    path, intervention_arms = info["path"], info["arms"]
    print("\n" + "#" * 80); print(f"# {exp_name}"); print("#" * 80)

    df = pd.read_csv(path, low_memory=False)
    df["Arm"] = df[ARM_COL].map(ARM_MAP).fillna(df[ARM_COL])
    df_int = df[df["Arm"].isin(intervention_arms)].copy()

    rows, dunn_store = [], {}
    for rating_col, rating_name in RATINGS.items():
        sub = df_int[["Arm", rating_col]].dropna().copy()
        desc = sub.groupby("Arm")[rating_col].agg(n="count", median="median", mean="mean").round(2).reindex(intervention_arms)
        print(f"\n{rating_name} (N={len(sub)})"); print(desc)

        groups = [sub.loc[sub["Arm"] == a, rating_col].values for a in intervention_arms]
        H, p_raw = kruskal(*groups)
        print(f"Kruskal-Wallis: H={H:.3f}, p={p_raw:.4f}")
        rows.append({"Dimension": rating_name, "H": H, "p_raw": p_raw})
        dunn_store[rating_name] = sub

    res = pd.DataFrame(rows)
    res["p_BH"] = false_discovery_control(res["p_raw"].values, method="bh")
    res["sig_BH"] = res["p_BH"] < 0.05
    print(f"\n{exp_name}: OMNIBUS WITH BH CORRECTION")
    print(res.round(4).to_string(index=False))

    for rating_name in res.loc[res["sig_BH"], "Dimension"]:
        sub = dunn_store[rating_name]
        rating_col = [k for k, v in RATINGS.items() if v == rating_name][0]
        dunn = sp.posthoc_dunn(sub, val_col=rating_col, group_col="Arm", p_adjust="holm")
        dunn = dunn.reindex(index=intervention_arms, columns=intervention_arms)
        print(f"\n{rating_name} Dunn-Holm post-hoc:")
        print(dunn.round(4).to_string())